# 주제 3 baseline — 주행 장면 인식 및 분할

부경대학교 교내 컴퓨터비전 부트캠프 · 최종 프로젝트 baseline · 2026. 8. 7.

---

도로 장면에서 차량·보행자·신호등을 탐지하고, **어떤 조건에서 무너지는지**를 체계적으로 정리합니다. 정답 라벨 없이도 할 수 있는 분석이 많은 주제입니다.

| STEP | 하는 일 |
|---|---|
| 0 · 1 | 환경 준비 · COCO 의 도로 장면만 골라 받기 |
| 2 | YOLO11 불러오기 |
| 3 | 전체 이미지에 탐지 |
| 4 | 결과를 표로 모으기 |
| 5 | **정량 분석** — 크기별 탐지율 · conf 분포 |
| 6 | **실패 사례** 고르고 저장 |

**시작 전에** — `런타임 → 런타임 유형 변경 → T4 GPU`. 그리고 STEP 1 의 다운로드 셀을
가장 먼저 실행해 두세요. 받는 동안 아래를 읽으면 됩니다.

## STEP 0 · 환경 준비

In [ ]:
!pip install -q ultralytics

In [ ]:
# 그래프 한글 폰트 (실패해도 실습에는 지장 없음)
try:
    !apt-get install -qq -y fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    import matplotlib.pyplot as plt
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
    print("한글 폰트 준비 완료")
except Exception as e:
    print("폰트 설치 건너뜀:", e)

In [ ]:
import os, glob, json, urllib.request
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
np.random.seed(0)


def show(img_bgr, title=None, w=9):
    h = w * img_bgr.shape[0] / img_bgr.shape[1]
    plt.figure(figsize=(w, h))
    plt.imshow(img_bgr[:, :, ::-1]); plt.axis("off")
    if title:
        plt.title(title, fontsize=12)
    plt.show()


def overlay(img_bgr, mask, color=(60, 120, 240), alpha=0.5):
    layer = np.zeros_like(img_bgr)
    layer[np.asarray(mask).astype(bool)] = color
    return cv2.addWeighted(img_bgr, 1.0, layer, alpha, 0)


def mask_iou(a, b):
    a = np.asarray(a).astype(bool)
    b = np.asarray(b).astype(bool)
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix = max(0, min(ax2, bx2) - max(ax1, bx1))
    iy = max(0, min(ay2, by2) - max(ay1, by1))
    inter = ix * iy
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / ua if ua > 0 else 0.0


def qbin(series, q, names):
    """값을 분위 구간으로 나눈다. 값이 적어 구간이 줄어도 죽지 않는다."""
    b = pd.qcut(series, q, duplicates="drop")
    cats = list(b.cat.categories)
    lab = names[:len(cats)] if len(cats) <= len(names) else [str(c) for c in cats]
    return b.cat.rename_categories(lab)


print("준비 완료")

In [ ]:
os.makedirs("outputs", exist_ok=True)     # 결과 이미지를 여기에 저장한다
os.makedirs("data", exist_ok=True)
print(os.listdir("."))

## STEP 1 · 데이터 준비 — COCO 도로 장면

`traffic light`, `car`, `person`, `bus`, `truck` 이 함께 등장하는 이미지만 골라 받습니다.

In [ ]:
N_IMAGES = 60
ROAD = ["car", "person", "traffic light", "bus", "truck", "bicycle", "motorcycle"]

if not os.path.exists("annotations/instances_val2017.json"):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o annotations_trainval2017.zip
print(os.path.exists("annotations/instances_val2017.json"))

In [ ]:
from pycocotools.coco import COCO
from collections import Counter

coco = COCO("annotations/instances_val2017.json")
tl = coco.getCatIds(catNms=["traffic light"])[0]
car = coco.getCatIds(catNms=["car"])[0]

# 신호등과 자동차가 같이 있는 이미지 = 대체로 도로 장면
ids = sorted(set(coco.getImgIds(catIds=[tl])) & set(coco.getImgIds(catIds=[car])))[:N_IMAGES]
paths = {}
for k, iid in enumerate(ids):
    info = coco.loadImgs(iid)[0]
    p = os.path.join("data", info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(info["coco_url"], p)
    paths[iid] = p
print("도로 장면", len(paths), "장")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, iid in zip(axes, list(paths)[:4]):
    ax.imshow(cv2.imread(paths[iid])[:, :, ::-1]); ax.axis("off")
plt.tight_layout(); plt.show()

## STEP 2 · 모델 불러오기

In [ ]:
from ultralytics import YOLO

det = YOLO("yolo11n.pt")
ROAD_IDS = [i for i, n in det.names.items() if n in ROAD]
print("관심 클래스 번호:", ROAD_IDS)

## STEP 3 · 전체 이미지에 탐지

conf 를 **낮게** 두고 전부 받아 둔 뒤, 분석 단계에서 걸러 냅니다.
이렇게 해야 threshold 를 바꿔 가며 비교할 수 있습니다.

In [ ]:
CONF_LOW = 0.05      # 일단 낮게 받아 둔다

rows = []
for k, (iid, p) in enumerate(paths.items()):
    r = det(p, conf=CONF_LOW, classes=ROAD_IDS, verbose=False)[0]
    H, W = r.orig_shape
    for b in r.boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        rows.append(dict(image=os.path.basename(p), image_id=iid,
                         cls=r.names[int(b.cls)], conf=round(float(b.conf), 4),
                         w=round(x2 - x1, 1), h=round(y2 - y1, 1),
                         area=round((x2 - x1) * (y2 - y1), 1),
                         rel_area=round((x2 - x1) * (y2 - y1) / (W * H), 5),
                         x1=round(x1, 1), y1=round(y1, 1), x2=round(x2, 1), y2=round(y2, 1)))
    if (k + 1) % 20 == 0:
        print(k + 1, "/", len(paths), flush=True)

COLS = ["image", "image_id", "cls", "conf", "w", "h", "area", "rel_area",
        "x1", "y1", "x2", "y2"]
df = pd.DataFrame(rows, columns=COLS)
if df.empty:
    print("탐지가 하나도 없습니다 — CONF_LOW 를 더 낮추거나 데이터를 확인하세요")
print(df.shape)
df.head()

## STEP 4 · 결과 표

In [ ]:
print("클래스별 (conf ≥ 0.25 기준)")
print(df[df.conf >= 0.25].cls.value_counts())
print()
print("이미지당 탐지 개수")
print(df[df.conf >= 0.25].groupby("image_id").size().describe().round(2))

## STEP 5 · 정량 분석 ★

정답 라벨이 없어도 이 정도는 말할 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ① 클래스별 confidence 분포
for c in df.cls.value_counts().head(4).index:
    axes[0].hist(df[df.cls == c].conf, bins=25, histtype="step", lw=2, label=c)
axes[0].axvline(0.25, color="#E97132", ls="--")
axes[0].legend(); axes[0].set_xlabel("confidence")
axes[0].set_title("클래스별 confidence 분포", fontsize=12)

# ② 크기(상대 면적)와 confidence
axes[1].scatter(df.rel_area, df.conf, s=8, alpha=0.4, color="#156082")
axes[1].set_xscale("log"); axes[1].set_xlabel("상대 면적 (로그)"); axes[1].set_ylabel("conf")
axes[1].set_title("작은 물체일수록 자신 없어지는가", fontsize=12)

# ③ conf threshold 를 올리면 몇 개가 남는가
ths = np.arange(0.05, 0.91, 0.05)
for c in ["car", "person", "traffic light"]:
    axes[2].plot(ths, [(df[(df.cls == c) & (df.conf >= t)].shape[0]) for t in ths],
                 marker="o", ms=3, label=c)
axes[2].legend(); axes[2].set_xlabel("conf threshold"); axes[2].set_ylabel("남는 탐지 수")
axes[2].set_title("threshold 를 올리면", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 크기 구간별 평균 confidence — "작은 물체에서 무너진다" 를 숫자로 확인
df["size_bin"] = qbin(df.rel_area, 4, ["아주 작음", "작음", "중간", "큼"])
t = df.pivot_table(index="size_bin", columns="cls", values="conf", aggfunc="mean")
print(t.round(3))
print()
print("크기 구간별, conf 0.25 를 넘는 비율 (%)")
print((df.groupby("size_bin").conf.apply(lambda s: (s >= 0.25).mean() * 100)).round(1))

### 여기에 여러분의 질문을 하나 더

- 이미지의 **밝기**(평균 픽셀값)와 탐지 개수는 관계가 있는가? — 야간·역광 가설
- 화면 **아래쪽(가까운 물체)** 과 **위쪽(먼 물체)** 의 confidence 차이는?
- `traffic light` 만 유난히 어려운가? 그렇다면 크기 때문인가 다른 이유인가?

In [ ]:
# 예시 — 이미지 밝기와 탐지 개수
bright = {iid: float(cv2.imread(p, cv2.IMREAD_GRAYSCALE).mean()) for iid, p in paths.items()}
cnt = df[df.conf >= 0.25].groupby("image_id").size()
xs = [bright[i] for i in cnt.index]
plt.figure(figsize=(6.5, 4))
plt.scatter(xs, cnt.values, s=20, color="#156082")
plt.xlabel("이미지 평균 밝기 (0~255)"); plt.ylabel("탐지 개수 (conf ≥ 0.25)")
plt.title("어두운 장면에서 덜 잡히는가", fontsize=12)
plt.tight_layout(); plt.show()
print("상관계수:", round(float(np.corrcoef(xs, cnt.values)[0, 1]), 3))

## STEP 6 · 실패 사례 ★

정답이 없으므로 **탐지가 적게 된 이미지**와 **애매한 예측이 많은 이미지**를 골라 봅니다.

In [ ]:
def draw(iid, tag, note=""):
    r = det(paths[iid], conf=0.25, classes=ROAD_IDS, verbose=False)[0]
    img = r.plot()
    cv2.imwrite(f"outputs/{tag}.png", img)
    show(img, f"{tag}  {note}", w=11)


cnt25 = df[df.conf >= 0.25].groupby("image_id").size()
gray = df[(df.conf >= 0.25) & (df.conf < 0.45)].groupby("image_id").size()

few = cnt25.sort_values().head(3).index          # 거의 못 찾은 장면
many_gray = gray.sort_values(ascending=False).head(3).index   # 애매한 예측이 많은 장면
good = cnt25.sort_values(ascending=False).head(3).index

for i, iid in enumerate(good, 1):
    draw(iid, f"success_{i}", f"— 탐지 {cnt25[iid]}건")
for i, iid in enumerate(few, 1):
    draw(iid, f"failure_few_{i}", f"— 탐지 {cnt25[iid]}건뿐, 밝기 {bright[iid]:.0f}")
for i, iid in enumerate(many_gray, 1):
    draw(iid, f"failure_gray_{i}", f"— 애매한 예측 {gray[iid]}건")
print("저장:", sorted(os.listdir("outputs")))

### 실패 사례 캡션 (직접 채우세요)

| 파일 | 무엇이 문제였나 | 근거가 되는 숫자 |
|---|---|---|
| failure_few_1.png | | 탐지 N건 / 밝기 M |
| failure_few_2.png | | |
| failure_gray_1.png | | 애매한 예측 N건 |

세 장의 **공통점**을 한 문장으로: 

## STEP 7 · 결과 이미지 내려받기

Colab 에서 `outputs/` 폴더를 통째로 압축해 내려받습니다.

In [ ]:
!zip -q -r outputs.zip outputs
try:
    from google.colab import files
    files.download("outputs.zip")
except Exception as e:
    print("Colab 이 아닙니다:", e)
print("저장된 이미지:", len(glob.glob("outputs/*.png")), "장")

## STEP 8 · 심화 트랙 (선택)

여기까지 여유 있게 끝냈다면 **`5일차_심화_finetuning_가이드.ipynb`** 를 여세요.
사전학습 모델을 그대로 쓰는 대신 **직접 fine-tuning 해서 같은 잣대로 비교**하는 트랙입니다.
평가에 **심화 10점**이 따로 배정되어 있습니다.

- 먼저 **위의 STEP 6 까지를 끝내 두어야** 합니다 — 기준선이 없으면 비교가 성립하지 않습니다
- T4 GPU 기준 30 epoch 에 10 ~ 15분
- **11:20 까지 여기에 도달하지 못했다면 심화는 포기하고 발표 준비로 넘어가세요**

심화는 fine-tuning 만 인정하는 것이 아닙니다. 새 지표를 직접 만들었거나, 데이터를 늘려
다시 쟀거나, prompt 전략을 체계적으로 비교했어도 같은 점수입니다.

---

## 제출 전 점검

- [ ] **런타임 → 런타임 다시 시작** 후 처음부터 끝까지 오류 없이 실행되는가
- [ ] `outputs/` 폴더에 성공 사례 3장 이상, 실패 사례 3장 이상
- [ ] 결과 이미지마다 캡션(파일명 또는 아래 마크다운 셀)이 붙어 있는가
- [ ] 표 또는 그래프가 하나 이상 있는가
- [ ] 아래 요약 다섯 줄을 채웠는가

## 결과 요약 (이 셀을 더블클릭해 직접 채우세요)

1. **무엇을 만들었나** —
2. **정량 결과** — (숫자 하나 이상)
3. **가장 잘 된 경우** —
4. **실패 유형과 개수** —
5. **시간이 더 있었다면** —
